# Четыре готовые модели

Выполняйте ячейки сверху вниз. Модели работают на CPU. Первый запуск скачивает зависимости и веса. Подробный разбор: docs/defense.md в репозитории. Проверка программ проведена локально; этот ноутбук нужно выполнить в своём сеансе Colab до защиты.

In [ ]:
import sys
print(sys.version)
assert (3, 11) <= sys.version_info[:2] <= (3, 12), "Нужен Python 3.11 или 3.12. Используйте локальный запуск с Python 3.12, если Colab сменил версию."


## Загрузить локальный проект

Код пока не опубликован. Эта ячейка предложит выбрать ml-homework.zip с компьютера и распакует его в сеансе. После перезапуска сеанса файлы обычно остаются, но при новом сеансе архив нужно загрузить заново.

In [ ]:
from pathlib import Path
import subprocess
import zipfile
project_dir = Path("/content/ml-engineering-1")
if not (project_dir / "text_task.py").exists():
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith(".zip")]
    assert len(archives) == 1, "Загрузите один файл ml-homework.zip"
    project_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            target = (project_dir / name).resolve()
            assert target.is_relative_to(project_dir.resolve()), "Некорректный путь в архиве"
        archive.extractall(project_dir)
%cd /content/ml-engineering-1


## Установить библиотеки

Эта ячейка занимает несколько минут. Здесь явно выбираются CPU-версии PyTorch. Если Colab предложит перезапуск сеанса, согласитесь, затем выполните следующую ячейку. После перезапуска установка повторно не нужна.

In [ ]:
%pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cpu
%pip install -q -r requirements.txt


In [ ]:
%cd /content/ml-engineering-1
import os
os.environ["USE_TF"] = "0"  # Transformers использует PyTorch, image_task отдельно импортирует TensorFlow.
os.environ["OMP_NUM_THREADS"] = "2"
import torch
import tensorflow as tf
import transformers
print("PyTorch", torch.__version__, "TensorFlow", tf.__version__, "Transformers", transformers.__version__)


## Демонстрационные файлы

Файлы скачиваются из официальных репозиториев PyTorch, Whisper и OpenCV. Для собственной проверки можно позже заменить их.

In [ ]:
%run download_samples.py


## Текст

Предположите класс до запуска. Затем сравните предсказание. score является оценкой конкретного ответа и не равен точности на наборе данных.

In [ ]:
from text_task import SentimentAnalyzer
from common import save_json
analyzer = SentimentAnalyzer()
text_result = analyzer.predict("Мне понравился этот фильм")
save_json(text_result, "results/text.json")
text_result


In [ ]:
%run evaluate_text.py


## Аудио

Демонстрационный файл содержит английскую речь, поэтому language="en". В следующем блоке можно загрузить свою русскую запись до 30 секунд.

In [ ]:
from IPython.display import Audio, display
display(Audio(filename="samples/jfk.flac"))
from audio_task import transcribe
audio_result = transcribe("samples/jfk.flac", language="en")
save_json(audio_result, "results/audio.json")
audio_result


In [ ]:
%run evaluate_audio.py


### Собственная русская запись

Этот блок необязателен для демонстрационного запуска, но полезен перед защитой. Запишите до 30 секунд спокойной речи и загрузите WAV или MP3. Вручную запишите точный эталон и сравните слова. Загружается именно ваш файл в Google Colab.

In [ ]:
# Уберите знаки # у строк ниже, чтобы загрузить и распознать свою запись.
# from google.colab import files
# uploaded = files.upload()
# own_file = next(iter(uploaded))
# own_result = transcribe(own_file, language="ru")
# save_json(own_result, "results/audio_ru.json")
# print(own_result)
# from metrics import word_error_rate
# reference = "Напишите здесь точную ручную расшифровку своей записи"
# print(word_error_rate(reference, own_result["text"]))


## Изображение

На входе фотография целиком. Сеть возвращает пять классов, без координат объектов.

In [ ]:
from IPython.display import Image
display(Image(filename="samples/dog.jpg", width=360))
from image_task import classify_image
image_result = classify_image("samples/dog.jpg")
save_json(image_result, "results/image.json")
image_result


## Видео

Для быстрого запуска обрабатываются первые 30 кадров. Порог снижен до 0.25: на этом фрагменте стандартный 0.5 не оставил рамок. При снижении порога возможны дополнительные ошибки. Записывается AVI без звука, JSON сохраняет рамки каждого кадра. Увеличьте max_frames при необходимости. На разных кадрах один человек может быть обнаружен снова.

In [ ]:
from video_task import detect_video
video_result = detect_video("samples/vtest.avi", max_frames=30, threshold=0.25)
save_json(video_result, "results/video.json")
{k: v for k, v in video_result.items() if k != "frames"}


### Посмотреть результат

AVI преобразуется в MP4 для показа в браузере. Это преобразование формата, а не новая обработка моделью.

In [ ]:
import imageio_ffmpeg
subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", "results/video.avi", "-vcodec", "libx264", "-pix_fmt", "yuv420p", "results/video_preview.mp4"], check=True, capture_output=True)
from IPython.display import Video
display(Video("results/video_preview.mp4", embed=True, width=640))


## Сохранить результаты

Скачайте архив результатов. После завершения сеанса Colab файлы могут исчезнуть. Числа в отчёте описывают локальную проверку; при описании своего запуска используйте именно свои JSON.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("my_results", "zip", "results")
files.download("my_results.zip")
